# Project Cycle 3 — Full Reproducible Analysis Notebook

**Research question:** Is the proportion of students who felt sad or hopeless different between female and male students?

This notebook contains the **complete workflow**: data loading, raw data checks, variable selection, recoding, cleaning, descriptive tables, two-sample inference, confidence intervals, effect size measures, alcohol-stratified extension, logistic regression, figures, and saved outputs.

Important: the original raw file is never modified. Recoded variables are created as new columns and saved only in `data/processed/`.

In [ ]:
# 0. Setup
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm

# Display settings
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

PROJECT_ROOT = Path('..').resolve() if Path('../data').exists() else Path.cwd()
# If this notebook is run from notebooks/, PROJECT_ROOT becomes project-cycle-3/.
# If it is run from project-cycle-3/, PROJECT_ROOT stays there.
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA = PROJECT_ROOT / 'data' / 'raw' / 'YRBS_2007.csv'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIG_DIR = PROJECT_ROOT / 'outputs' / 'figures'
TABLE_DIR = PROJECT_ROOT / 'outputs' / 'tables'
SUMMARY_DIR = PROJECT_ROOT / 'outputs' / 'summary'

for d in [PROCESSED_DIR, FIG_DIR, TABLE_DIR, SUMMARY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Raw data exists:', RAW_DATA.exists(), RAW_DATA)

## 1. Load raw data

The raw data must remain unchanged in `data/raw/`. All cleaning and recoding are done in memory first, then saved as a new processed CSV.

In [ ]:
df_raw = pd.read_csv(RAW_DATA)
print('Raw shape:', df_raw.shape)
df_raw.head()

## 2. Select variables for the approved Cycle 3 question and extension

Main approved question: **Question 3 — Gender and Sad or Hopeless Feeling**

- Group variable: `WhatIsYourSex`
- Response variable: `SadOrHopeless`

Creative extension variable:

- `CurrentAlcoholUse` is used as an additional explanatory factor in stratified summaries and logistic regression.
- This does not replace the required two-sample inference; it adds context.

In [ ]:
needed_cols = ['WhatIsYourSex', 'SadOrHopeless', 'CurrentAlcoholUse']
missing_cols = [c for c in needed_cols if c not in df_raw.columns]
if missing_cols:
    raise ValueError(f'Missing required columns: {missing_cols}')

df = df_raw[needed_cols].copy()
print(df.shape)
df.head()

## 3. Raw coding checks

Before recoding, check the original values and missing values. This is important because Cycle 3 warns against treating coded categories as continuous numbers or ignoring missing/invalid codes.

In [ ]:
for col in needed_cols:
    print('\n' + '='*70)
    print(col)
    print(df[col].value_counts(dropna=False).sort_index())

## 4. Recoding rules

The notebook keeps the original variables and creates new analysis variables.

### Sex
- `WhatIsYourSex = 1` → Female
- `WhatIsYourSex = 2` → Male

### Sad or Hopeless
- `SadOrHopeless = 1` → Yes / success / 1
- `SadOrHopeless = 2` → No / failure / 0

### Current Alcohol Use
- `CurrentAlcoholUse = 1` → No current alcohol use / 0
- `CurrentAlcoholUse = 2–7` → Current alcohol use / 1

The alcohol variable is not used to change the original response. It is used only for supplementary stratified analysis and logistic regression.

In [ ]:
def recode_sex(x):
    if x == 1:
        return 'Female'
    if x == 2:
        return 'Male'
    return np.nan

def recode_yes_no_from_1_2(x):
    if x == 1:
        return 1
    if x == 2:
        return 0
    return np.nan

def recode_current_alcohol_binary(x):
    if x == 1:
        return 0
    if x in [2, 3, 4, 5, 6, 7]:
        return 1
    return np.nan

df['sex'] = df['WhatIsYourSex'].apply(recode_sex)
df['sad_hopeless'] = df['SadOrHopeless'].apply(recode_yes_no_from_1_2)
df['current_alcohol_use'] = df['CurrentAlcoholUse'].apply(recode_current_alcohol_binary)

df['sad_hopeless_label'] = df['sad_hopeless'].map({1: 'Yes', 0: 'No'})
df['current_alcohol_use_label'] = df['current_alcohol_use'].map({1: 'Current alcohol use', 0: 'No current alcohol use'})

print(df[['WhatIsYourSex','sex','SadOrHopeless','sad_hopeless','CurrentAlcoholUse','current_alcohol_use']].head(10))

## 5. Cleaning: remove rows that cannot be used for the selected analysis

For the main required two-sample inference, we need non-missing `sex` and `sad_hopeless`.

For the alcohol extension, we additionally need non-missing `current_alcohol_use`. The processed data saved below includes all three recoded variables.

In [ ]:
main_clean = df.dropna(subset=['sex', 'sad_hopeless']).copy()
extension_clean = df.dropna(subset=['sex', 'sad_hopeless', 'current_alcohol_use']).copy()

# Convert numeric recoded columns to integers after missing values are removed.
for tmp in [main_clean, extension_clean]:
    tmp['sad_hopeless'] = tmp['sad_hopeless'].astype(int)
    if 'current_alcohol_use' in tmp.columns:
        tmp['current_alcohol_use'] = tmp['current_alcohol_use'].astype('Int64')

processed_path = PROCESSED_DIR / 'yrbs_cycle3_cleaned_recoded.csv'
extension_clean.to_csv(processed_path, index=False)

print('Main analysis rows:', len(main_clean))
print('Extension analysis rows:', len(extension_clean))
print('Processed data saved to:', processed_path)

## 6. Descriptive summary by sex

The key descriptive statistic is the proportion of students in each sex group who answered Yes to `SadOrHopeless`.

In [ ]:
summary = (main_clean
    .groupby('sex')
    .agg(
        n=('sad_hopeless', 'size'),
        sad_yes=('sad_hopeless', 'sum'),
        sad_proportion=('sad_hopeless', 'mean')
    )
    .reset_index())

# Force the comparison order used throughout the report: Female minus Male.
summary['sex'] = pd.Categorical(summary['sex'], categories=['Female','Male'], ordered=True)
summary = summary.sort_values('sex').reset_index(drop=True)
summary['sad_percent'] = summary['sad_proportion'] * 100
summary_path = TABLE_DIR / 'group_summary_by_sex.csv'
summary.to_csv(summary_path, index=False)
summary

## 7. Visualization 1 — proportion by sex

This figure directly supports the main two-sample comparison.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(summary['sex'].astype(str), summary['sad_percent'])
ax.set_ylabel('Students reporting sad or hopeless feelings (%)')
ax.set_xlabel('Sex')
ax.set_title('Sad or Hopeless Feelings by Sex')
ax.set_ylim(0, max(summary['sad_percent']) + 10)
for i, row in summary.iterrows():
    ax.text(i, row['sad_percent'] + 1, f"{row['sad_percent']:.1f}%\n(n={int(row['n'])})", ha='center', va='bottom')
fig.tight_layout()
fig_path = FIG_DIR / '01_proportion_by_sex.png'
fig.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', fig_path)

## 8. Main two-sample inference: difference in proportions

The main parameter is:

$$p_{Female} - p_{Male}$$

where $p$ is the population proportion of students reporting sad or hopeless feelings.

### Hypotheses

- $H_0: p_{Female} - p_{Male} = 0$
- $H_a: p_{Female} - p_{Male} \ne 0$

The required Cycle 3 method for a binary response is a two-proportion z-test. To make the analysis less plain, we also report confidence interval, risk difference, relative risk, odds ratio, and logistic regression results.

In [ ]:
# Extract counts in fixed order.
female = summary.loc[summary['sex'].astype(str) == 'Female'].iloc[0]
male = summary.loc[summary['sex'].astype(str) == 'Male'].iloc[0]

x1, n1 = int(female['sad_yes']), int(female['n'])
x2, n2 = int(male['sad_yes']), int(male['n'])
p1, p2 = x1/n1, x2/n2
diff = p1 - p2

# Two-proportion z-test under H0: p1 = p2.
p_pool = (x1 + x2) / (n1 + n2)
se_null = math.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))
z_stat = diff / se_null
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))

# 95% CI for the difference in proportions using unpooled SE.
se_diff = math.sqrt(p1*(1-p1)/n1 + p2*(1-p2)/n2)
zcrit = stats.norm.ppf(0.975)
ci_low = diff - zcrit * se_diff
ci_high = diff + zcrit * se_diff

# Additional effect measures.
risk_difference = diff
relative_risk = p1 / p2
odds_ratio = (x1 / (n1 - x1)) / (x2 / (n2 - x2))
log_or = math.log(odds_ratio)
se_log_or = math.sqrt(1/x1 + 1/(n1-x1) + 1/x2 + 1/(n2-x2))
or_ci_low = math.exp(log_or - zcrit*se_log_or)
or_ci_high = math.exp(log_or + zcrit*se_log_or)

inference = pd.DataFrame([{
    'comparison': 'Female - Male',
    'female_n': n1,
    'male_n': n2,
    'female_sad_yes': x1,
    'male_sad_yes': x2,
    'female_proportion': p1,
    'male_proportion': p2,
    'difference_in_proportions': diff,
    'ci_95_low': ci_low,
    'ci_95_high': ci_high,
    'z_statistic': z_stat,
    'p_value': p_value,
    'risk_difference': risk_difference,
    'relative_risk': relative_risk,
    'odds_ratio': odds_ratio,
    'odds_ratio_ci_95_low': or_ci_low,
    'odds_ratio_ci_95_high': or_ci_high,
    'alpha': 0.05,
    'decision': 'Reject H0' if p_value < 0.05 else 'Fail to reject H0'
}])

inference_path = TABLE_DIR / 'inference_summary_table.csv'
inference.to_csv(inference_path, index=False)
inference.T

## 9. Visualization 2 — effect-size dashboard

The 95% confidence interval is kept in the summary table, but the slide uses a more infographic-style effect-size dashboard instead of a plain CI-only plot. This makes the inference easier to explain: direction, size, and practical meaning are shown together.

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 4.2))
ax.axis('off')

card_data = [
    ('Risk difference', f'{risk_difference*100:.1f} pp', 'Female proportion − Male proportion'),
    ('Relative risk', f'{relative_risk:.2f}×', 'Female proportion / Male proportion'),
    ('Odds ratio', f'{odds_ratio:.2f}×', 'Female odds / Male odds'),
]

for i, (label, value, subtitle) in enumerate(card_data):
    x0 = 0.04 + i*0.32
    rect = plt.Rectangle((x0, 0.25), 0.28, 0.55, fill=False, linewidth=1.5, transform=ax.transAxes)
    ax.add_patch(rect)
    ax.text(x0+0.14, 0.66, label, ha='center', va='center', fontsize=12, weight='bold', transform=ax.transAxes)
    ax.text(x0+0.14, 0.50, value, ha='center', va='center', fontsize=24, weight='bold', transform=ax.transAxes)
    ax.text(x0+0.14, 0.36, subtitle, ha='center', va='center', fontsize=8.5, transform=ax.transAxes)

ax.text(0.5, 0.92, 'Effect-Size Dashboard for Female vs Male Comparison', ha='center', va='center', fontsize=14, weight='bold', transform=ax.transAxes)
ax.text(0.5, 0.12, f'Two-proportion z-test: z = {z_stat:.2f}, p < 0.001; 95% CI for difference is saved in the summary table.', ha='center', va='center', fontsize=10, transform=ax.transAxes)

fig.tight_layout()
fig_path = FIG_DIR / '02_effect_size_dashboard.png'
fig.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', fig_path)

## 10. Extension: alcohol-stratified summary

This section addresses the request to consider current alcohol use while keeping the approved two-sample question intact.

This is not a new approved research question; it is a supplementary analysis that asks whether the female-male difference is visible within alcohol-use strata.

In [ ]:
stratified = (extension_clean
    .groupby(['current_alcohol_use_label', 'sex'])
    .agg(
        n=('sad_hopeless', 'size'),
        sad_yes=('sad_hopeless', 'sum'),
        sad_proportion=('sad_hopeless', 'mean')
    )
    .reset_index())
stratified['sad_percent'] = stratified['sad_proportion'] * 100
stratified_path = TABLE_DIR / 'stratified_summary_by_sex_and_alcohol.csv'
stratified.to_csv(stratified_path, index=False)
stratified

In [ ]:
pivot = stratified.pivot(index='current_alcohol_use_label', columns='sex', values='sad_percent')
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(pivot.index))
width = 0.35
for i, sex in enumerate(['Female','Male']):
    values = pivot[sex].values
    ax.bar(x + (i-0.5)*width, values, width, label=sex)
    for j, v in enumerate(values):
        ax.text(x[j] + (i-0.5)*width, v + 1, f'{v:.1f}%', ha='center', va='bottom', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(pivot.index, rotation=0)
ax.set_ylabel('Sad or hopeless (%)')
ax.set_title('Sad or Hopeless Feelings by Sex and Current Alcohol Use')
ax.legend(title='Sex')
ax.set_ylim(0, max(stratified['sad_percent']) + 12)
fig.tight_layout()
fig_path = FIG_DIR / '03_stratified_by_alcohol.png'
fig.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', fig_path)

## 11. Mantel-Haenszel adjusted odds ratio by alcohol status

This adds a more research-style effect estimate: the association between sex and sad/hopeless feeling adjusted for alcohol-use strata.

In [ ]:
# Build stratum-specific 2x2 tables for Female vs Male and Sad Yes vs No.
rows = []
for alcohol_label, g in extension_clean.groupby('current_alcohol_use_label'):
    a = int(((g['sex'] == 'Female') & (g['sad_hopeless'] == 1)).sum())
    b = int(((g['sex'] == 'Female') & (g['sad_hopeless'] == 0)).sum())
    c = int(((g['sex'] == 'Male') & (g['sad_hopeless'] == 1)).sum())
    d = int(((g['sex'] == 'Male') & (g['sad_hopeless'] == 0)).sum())
    n = a + b + c + d
    rows.append({'alcohol_status': alcohol_label, 'a_female_yes': a, 'b_female_no': b, 'c_male_yes': c, 'd_male_no': d, 'n': n, 'stratum_or': (a*d)/(b*c)})

mh_table = pd.DataFrame(rows)
# Mantel-Haenszel common OR approximation.
num = sum((r.a_female_yes * r.d_male_no) / r.n for r in mh_table.itertuples())
den = sum((r.b_female_no * r.c_male_yes) / r.n for r in mh_table.itertuples())
mh_or = num / den
mh_table['mantel_haenszel_common_or'] = mh_or
mh_path = TABLE_DIR / 'mantel_haenszel_by_alcohol.csv'
mh_table.to_csv(mh_path, index=False)
mh_table

## 12. Logistic regression extension

Model:

$$logit(P(SadOrHopeless=1)) = \beta_0 + \beta_1 Female + \beta_2 CurrentAlcoholUse$$

This does **not** replace the required two-proportion z-test. It adds an adjusted effect-size interpretation.

In [ ]:
model_df = extension_clean.copy()
model_df['female'] = (model_df['sex'] == 'Female').astype(int)
model_df['alcohol_yes'] = model_df['current_alcohol_use'].astype(int)

X = sm.add_constant(model_df[['female', 'alcohol_yes']])
y = model_df['sad_hopeless'].astype(int)
logit_model = sm.Logit(y, X).fit(disp=False)
print(logit_model.summary())

params = logit_model.params
conf = logit_model.conf_int()
or_table = pd.DataFrame({
    'term': params.index,
    'coef': params.values,
    'odds_ratio': np.exp(params.values),
    'ci_95_low': np.exp(conf[0].values),
    'ci_95_high': np.exp(conf[1].values),
    'p_value': logit_model.pvalues.values
})
or_path = TABLE_DIR / 'logistic_regression_odds_ratios.csv'
or_table.to_csv(or_path, index=False)
or_table

In [ ]:
plot_or = or_table[or_table['term'].isin(['female', 'alcohol_yes'])].copy()
plot_or['label'] = plot_or['term'].map({'female':'Female vs Male', 'alcohol_yes':'Current alcohol use vs none'})
fig, ax = plt.subplots(figsize=(8, 4))
ypos = np.arange(len(plot_or))
ax.errorbar(plot_or['odds_ratio'], ypos,
            xerr=[plot_or['odds_ratio'] - plot_or['ci_95_low'], plot_or['ci_95_high'] - plot_or['odds_ratio']],
            fmt='o', capsize=5)
ax.axvline(1, linestyle='--', linewidth=1)
ax.set_yticks(ypos)
ax.set_yticklabels(plot_or['label'])
ax.set_xlabel('Adjusted odds ratio, 95% CI')
ax.set_title('Logistic Regression Effect Estimates')
for i, row in plot_or.reset_index(drop=True).iterrows():
    ax.text(row['ci_95_high'] + 0.05, i, f"OR={row['odds_ratio']:.2f}", va='center')
fig.tight_layout()
fig_path = FIG_DIR / '04_odds_ratio_forest.png'
fig.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', fig_path)

## 13. Predicted probabilities from logistic regression

This makes the adjusted model easier to explain visually.

In [ ]:
pred_grid = pd.DataFrame({
    'female': [0, 1, 0, 1],
    'alcohol_yes': [0, 0, 1, 1]
})
pred_grid = sm.add_constant(pred_grid, has_constant='add')
pred = logit_model.get_prediction(pred_grid).summary_frame(alpha=0.05)
pred_grid['sex'] = pred_grid['female'].map({1:'Female', 0:'Male'})
pred_grid['alcohol_status'] = pred_grid['alcohol_yes'].map({1:'Current alcohol use', 0:'No current alcohol use'})
pred_grid['predicted_probability'] = pred['predicted']
pred_grid['ci_95_low'] = pred['ci_lower']
pred_grid['ci_95_high'] = pred['ci_upper']
pred_out = pred_grid[['sex','alcohol_status','predicted_probability','ci_95_low','ci_95_high']]
pred_path = TABLE_DIR / 'model_predicted_probabilities.csv'
pred_out.to_csv(pred_path, index=False)
pred_out

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
pred_out['group'] = pred_out['sex'] + '\n' + pred_out['alcohol_status']
y = pred_out['predicted_probability'] * 100
yerr_lower = (pred_out['predicted_probability'] - pred_out['ci_95_low']) * 100
yerr_upper = (pred_out['ci_95_high'] - pred_out['predicted_probability']) * 100
ax.bar(pred_out['group'], y)
ax.errorbar(pred_out['group'], y, yerr=[yerr_lower, yerr_upper], fmt='none', capsize=5)
ax.set_ylabel('Adjusted predicted probability (%)')
ax.set_title('Adjusted Predicted Probability of Sad or Hopeless Feelings')
ax.set_ylim(0, max(y + yerr_upper) + 10)
for i, val in enumerate(y):
    ax.text(i, val + 1, f'{val:.1f}%', ha='center')
fig.tight_layout()
fig_path = FIG_DIR / '05_logistic_predicted_probabilities.png'
fig.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', fig_path)

## 14. Type I and Type II error discussion

These are included as interpretation notes, not as additional calculations.

- **Type I error:** concluding that female and male students have different proportions of sad or hopeless feelings when there is actually no real population difference.
- **Type II error:** failing to detect a real female-male difference in sad or hopeless feelings.

Because the sample size is large, the analysis has strong ability to detect differences. However, large samples can make even small differences statistically significant, so the report also emphasizes the size of the difference, confidence interval, and odds ratios.

## 15. Final interpretation

In [ ]:
final_text = f'''Research question: Is the proportion of students who felt sad or hopeless different between female and male students?

Female students had a sad/hopeless proportion of {p1:.3f} ({p1*100:.1f}%), while male students had a proportion of {p2:.3f} ({p2*100:.1f}%). The estimated difference was {diff:.3f}, or {diff*100:.1f} percentage points, with a 95% confidence interval from {ci_low*100:.1f} to {ci_high*100:.1f} percentage points. The two-proportion z-test gave z = {z_stat:.2f} and p = {p_value:.3e}. At alpha = 0.05, we reject the null hypothesis.

In context, female students in this dataset reported sad or hopeless feelings at a higher proportion than male students. The estimated odds ratio was {odds_ratio:.2f}. In the supplementary logistic regression that adjusted for current alcohol use, the adjusted odds ratio for female versus male students remained above 1, meaning the female-male difference was still visible after accounting for alcohol use.

Because this is observational survey data, the result should be interpreted as an association, not causation. The analysis does not prove that sex causes sad or hopeless feelings, and self-reported survey responses may contain reporting or recall bias.
'''
print(final_text)
(SUMMARY_DIR / 'final_summary.txt').write_text(final_text, encoding='utf-8')

## 16. Save key numbers for slide/report generation

In [ ]:
key_numbers = {
    'main_n': int(len(main_clean)),
    'extension_n': int(len(extension_clean)),
    'female_n': int(n1),
    'male_n': int(n2),
    'female_percent': round(p1*100, 1),
    'male_percent': round(p2*100, 1),
    'difference_percentage_points': round(diff*100, 1),
    'ci_low_percentage_points': round(ci_low*100, 1),
    'ci_high_percentage_points': round(ci_high*100, 1),
    'z_statistic': round(z_stat, 2),
    'p_value': float(p_value),
    'odds_ratio': round(odds_ratio, 2),
    'odds_ratio_ci_low': round(or_ci_low, 2),
    'odds_ratio_ci_high': round(or_ci_high, 2),
    'mantel_haenszel_common_or': round(mh_or, 2),
    'logistic_female_adjusted_or': round(float(or_table.loc[or_table['term']=='female','odds_ratio'].iloc[0]), 2),
    'logistic_alcohol_adjusted_or': round(float(or_table.loc[or_table['term']=='alcohol_yes','odds_ratio'].iloc[0]), 2)
}
key_path = TABLE_DIR / 'key_numbers.json'
key_path.write_text(json.dumps(key_numbers, indent=2), encoding='utf-8')
key_numbers

## 17. Requirement checklist

- Clear two-sample research question: yes
- Correct group variable and response variable: yes
- Data cleaning and recoding documented: yes
- Suitable two-sample method: two-proportion z-test
- Confidence interval for difference: yes
- Hypothesis test and conclusion: yes
- At least two meaningful visualizations: yes
- Clear summary table: yes
- Interpretation in context: yes
- One-slide infographic summary: saved separately in `report/`
- Alcohol extension: included as supplementary analysis without modifying raw data
